# 09 Silver Immunization Clean

## Purpose

This notebook creates the Silver Immunization table from raw FHIR Immunization resources.

## What We Are Doing

We will:
1. Create/read `healthcare_catalog.bronze.immunization_raw`
2. Extract vaccine-related fields
3. Clean patient and encounter IDs
4. Convert vaccine dates into timestamp format
5. Save the clean table into the Silver layer

## Why We Are Doing This

Immunization data is important for:
- vaccine history
- preventive care analytics
- population health dashboards
- clinical risk scoring
- quality measure analysis

## Expected Final Output

A clean Delta table:

`healthcare_catalog.silver.immunization_clean`

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
We need these functions to filter, explode, extract, clean, and transform FHIR data.

### Expected Output
PySpark functions are available in this notebook.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Immunization Table

### What We Are Doing
We are loading the Bronze Immunization table.

### Why We Are Doing This
The Bronze layer stores raw FHIR Immunization resources.

### Expected Output
A DataFrame named:

`immunization_raw_df`

In [0]:
immunization_raw_df = spark.table(
    "healthcare_catalog.bronze.immunization_raw"
)

print("Bronze immunization_raw table loaded successfully.")

Bronze immunization_raw table loaded successfully.


## Step 3 — Inspect Raw Immunization Schema

### What We Are Doing
We are printing the schema of the raw Immunization resource.

### Why We Are Doing This
FHIR resources are deeply nested JSON structures.

We need to identify:
- vaccine fields
- patient references
- encounter references
- timestamps
- vaccine metadata

### Expected Output
FHIR Immunization schema.

In [0]:
immunization_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Immunization Columns

### What We Are Doing
We are extracting vaccine-related fields from nested FHIR Immunization resources.

### Why We Are Doing This
Analytics and ML models require flattened structured tables.

### Fields We Will Extract

- immunization_id
- patient_reference
- encounter_reference
- vaccine_name
- vaccine_code
- vaccine_display
- immunization_status
- occurrence_datetime
- primary_source
- lot_number

### Expected Output
A flattened DataFrame:

`immunization_clean_df`

In [0]:
immunization_clean_df = immunization_raw_df.select(

    col("resource.id").alias("immunization_id"),

    col("resource.patient.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    col("resource.vaccineCode.text").alias("vaccine_name"),

    col("resource.vaccineCode.coding")[0]["code"].alias("vaccine_code"),

    col("resource.vaccineCode.coding")[0]["display"].alias("vaccine_display"),

    col("resource.status").alias("immunization_status"),

    col("resource.occurrenceDateTime").alias("occurrence_datetime"),

    col("resource.primarySource").alias("primary_source"),

    col("resource.lotNumber").alias("lot_number")
)

print("Immunization clean DataFrame created successfully.")

Immunization clean DataFrame created successfully.


## Step 5 — Convert Immunization Timestamp

### What We Are Doing
We are converting occurrence_datetime into Spark timestamp format.

### Why We Are Doing This
Timestamp format is required for:
- time-series analytics
- vaccination timelines
- public health reporting

### Expected Output
Timestamp conversion completed successfully.

In [0]:
immunization_clean_df = immunization_clean_df.withColumn(
    "occurrence_datetime",
    to_timestamp(col("occurrence_datetime"))
)

print("Immunization timestamp converted successfully.")

Immunization timestamp converted successfully.


## Step 6 — Extract Clean Patient and Encounter IDs

### What We Are Doing
We are removing `urn:uuid:` from FHIR references.

### Why We Are Doing This
Clean IDs are required for joining healthcare tables together.

### Expected Output
New columns:
- patient_id
- encounter_id

In [0]:
immunization_clean_df = immunization_clean_df.withColumn(
    "patient_id",
    regexp_extract(col("patient_reference"), r"urn:uuid:(.*)", 1)
)

immunization_clean_df = immunization_clean_df.withColumn(
    "encounter_id",
    regexp_extract(col("encounter_reference"), r"urn:uuid:(.*)", 1)
)

print("Patient and encounter IDs extracted successfully.")

Patient and encounter IDs extracted successfully.


## Step 7 — Inspect Clean Immunization Data

### What We Are Doing
We are displaying the clean immunization table.

### Why We Are Doing This
We need to verify:
- vaccine names
- timestamps
- IDs
- statuses

### Expected Output
A clean vaccine-level healthcare table.

In [0]:
display(immunization_clean_df)

immunization_id,patient_reference,encounter_reference,vaccine_name,vaccine_code,vaccine_display,immunization_status,occurrence_datetime,primary_source,lot_number,patient_id,encounter_id
9c58d129-d3b9-6d13-327c-128929200660,urn:uuid:d7c46304-29f5-5ddb-f5df-7d816cf4f318,urn:uuid:6debb662-0b81-0dab-9f3e-fd5740b490cf,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2014-10-14T09:19:08.000Z,true,null,d7c46304-29f5-5ddb-f5df-7d816cf4f318,6debb662-0b81-0dab-9f3e-fd5740b490cf
5aa4d544-9a87-70da-413f-c4a892877c83,urn:uuid:d7c46304-29f5-5ddb-f5df-7d816cf4f318,urn:uuid:6debb662-0b81-0dab-9f3e-fd5740b490cf,Td (adult) preservative free,113,Td (adult) preservative free,completed,2014-10-14T09:19:08.000Z,true,null,d7c46304-29f5-5ddb-f5df-7d816cf4f318,6debb662-0b81-0dab-9f3e-fd5740b490cf
b38521c4-62ec-4175-cd90-ab1212a878af,urn:uuid:d7c46304-29f5-5ddb-f5df-7d816cf4f318,urn:uuid:6dc48801-821c-76a9-7b69-dab6da2ccba5,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2017-10-17T09:19:08.000Z,true,null,d7c46304-29f5-5ddb-f5df-7d816cf4f318,6dc48801-821c-76a9-7b69-dab6da2ccba5
b8c1f1da-0ca3-65d8-c74d-db92a4e7b370,urn:uuid:d7c46304-29f5-5ddb-f5df-7d816cf4f318,urn:uuid:bb148dcd-0fc3-364c-c39a-b5a9b93ee07a,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2020-10-20T09:19:08.000Z,true,null,d7c46304-29f5-5ddb-f5df-7d816cf4f318,bb148dcd-0fc3-364c-c39a-b5a9b93ee07a
da502283-a438-80ee-281a-3bcf735ab6dc,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:fd2f267b-6acd-b9ce-fc3d-69612527befd,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2014-06-10T12:03:04.000Z,true,null,988ba5c5-bb2c-1453-cfe7-16ea41c47b42,fd2f267b-6acd-b9ce-fc3d-69612527befd
4a172762-8d95-9c89-1c69-f2ae04eb724c,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:c5bb234e-95ed-a577-8cc8-fb7df69e42f2,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2017-06-13T12:03:04.000Z,true,null,988ba5c5-bb2c-1453-cfe7-16ea41c47b42,c5bb234e-95ed-a577-8cc8-fb7df69e42f2
78088c69-5425-011a-7501-625302022bbe,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:e155ab5d-5f4a-f71d-42e1-9a13a511d35f,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2019-11-12T12:03:04.000Z,true,null,988ba5c5-bb2c-1453-cfe7-16ea41c47b42,e155ab5d-5f4a-f71d-42e1-9a13a511d35f
34592c48-c160-21d7-25b1-11818ccc273a,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:b18533ed-fd3a-c765-691f-1a7cffeb2a7e,"Influenza, seasonal, injectable, preservative free",140,"Influenza, seasonal, injectable, preservative free",completed,2020-03-31T12:03:04.000Z,true,null,988ba5c5-bb2c-1453-cfe7-16ea41c47b42,b18533ed-fd3a-c765-691f-1a7cffeb2a7e
cd1ecd4d-0048-dc1e-868c-8e0ba4d632e5,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:b18533ed-fd3a-c765-691f-1a7cffeb2a7e,"Hep A, adult",52,"Hep A, adult",completed,2020-03-31T12:03:04.000Z,true,null,988ba5c5-bb2c-1453-cfe7-16ea41c47b42,b18533ed-fd3a-c765-691f-1a7cffeb2a7e
2a1f4a9b-9a1c-6f3a-d083-42203ba421dc,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:8bb71cb6-4284-7f84-2432-9692ba2e1349,"SARS-COV-2 (COVID-19) vaccine, mRNA, spike protein, LNP, preservative free, 30 mcg/0.3mL dose",208,"SARS-COV-2 (COVID-19) vaccine, mRNA, spike protein, LNP, preservative free, 30 mcg/0.3mL dose",completed,2021-08-31T12:03:04.000Z,true,null,988ba5c5-bb2c-1453-cfe7-16ea41c47b42,8bb71cb6-4284-7f84-2432-9692ba2e1349


## Step 8 — Check Most Common Vaccines

### What We Are Doing
We are counting vaccine frequency.

### Why We Are Doing This
This helps us understand:
- vaccination patterns
- preventive care coverage
- population health trends

### Expected Output
Top vaccine frequency table.

In [0]:
display(

    immunization_clean_df.groupBy(
        "vaccine_name"
    ).count().orderBy(
        desc("count")
    )

)

vaccine_name,count
"Influenza, seasonal, injectable, preservative free",4452
Td (adult) preservative free,391
"SARS-COV-2 (COVID-19) vaccine, mRNA, spike protein, LNP, preservative free, 30 mcg/0.3mL dose",371
DTaP,292
Pneumococcal conjugate PCV 13,270
"SARS-COV-2 (COVID-19) vaccine, mRNA, spike protein, LNP, preservative free, 100 mcg/0.5mL dose",252
IPV,241
"HPV, quadrivalent",229
meningococcal MCV4P,219
Hib (PRP-OMP),175


## Step 9 — Check Null Values

### What We Are Doing
We are checking missing values in the immunization table.

### Why We Are Doing This
Silver validation ensures high-quality analytics-ready data.

### Expected Output
A null-count summary table.

In [0]:
display(

    immunization_clean_df.select(

        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)

            for column_name in immunization_clean_df.columns
        ]

    )

)

immunization_id,patient_reference,encounter_reference,vaccine_name,vaccine_code,vaccine_display,immunization_status,occurrence_datetime,primary_source,lot_number,patient_id,encounter_id
0,0,0,0,0,0,0,0,0,8100,0,0


## Step 10 — Save Silver Immunization Table

### What We Are Doing
We are saving the clean immunization table into the Silver layer.

### Why We Are Doing This
The Silver layer stores:
- cleaned
- normalized
- analytics-ready

vaccination data.

### Expected Output
A Delta table:

`healthcare_catalog.silver.immunization_clean`

In [0]:
immunization_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.immunization_clean")

print("Silver immunization_clean table saved successfully.")

Silver immunization_clean table saved successfully.


## Step 11 — Verify Silver Tables

### What We Are Doing
We are listing all Silver tables.

### Why We Are Doing This
We want to confirm that the new immunization table was saved successfully.

### Expected Output
`immunization_clean` should appear in the Silver table list.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+------------------------+-----------+
|database|tableName               |isTemporary|
+--------+------------------------+-----------+
|silver  |condition_clean         |false      |
|silver  |encounter_clean         |false      |
|silver  |immunization_clean      |false      |
|silver  |medication_request_clean|false      |
|silver  |observation_clean       |false      |
|silver  |patient_clean           |false      |
|silver  |procedure_clean         |false      |
+--------+------------------------+-----------+

